# 39. tSZ power spectrum per yang26 rotation group (L2p8_m9 lc0)

Unlensed per-shell Compton-$y$ maps are summed **within each rotation group**
(no yang26 rotation applied to the sums; grouping follows `ANGLES_L2P8` in
`build_y_map.py`).  For each group map we measure $D_\ell$ and compare to
**hmfast** (nb05: A10 GNFW, $B=1.0$, D3A cosmology) integrated over that
group's $z$ interval.

| group | shells | $z$ range | rotation $(\theta, \phi)$ rad | $\sum y$ |
|-------|--------|-----------|----------------------------------|----------|
| 0 | 0–6 | 0.001 – 0.35 | identity | 68.67 |
| 1 | 7–15 | 0.35 – 0.80 | 2.118, 0.964 | 81.41 |
| 2 | 16–26 | 0.80 – 1.35 | 1.291, 1.748 | 72.95 |
| 3 | 27–43 | 1.35 – 2.20 | 5.692, 0.563 | 61.42 |
| 4 | 44–59 | 2.20 – 3.00 | 3.797, 1.455 | 24.80 |

Maps: `/rds/rds-lxu/flamingo/L2p8_m9/lightcone0/healpix_map/rotation_groups/`.

Pipeline (nb05 / nb38): monopole subtract, `anafast`, pixel-window deconv,
$/f_\mathrm{sky}$.  Theory: `z = jnp.geomspace(zlo, zhi, 60)` per group.

In [ ]:
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("JAX_PLATFORMS", "cpu")

REPO = Path("/scratch/scratch-lxu/flamingo_repo")
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
import jax.numpy as jnp

from flamingo.catalogue import D3A_COSMOLOGY
from hmfast.halos import HaloModel
from hmfast.halos.profiles import GNFWPressureProfile
from hmfast.tracers import tSZTracer

plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 11,
    "legend.fontsize": 8,
    "figure.dpi": 100,
    "savefig.dpi": 300,
})

MAP_DIR = Path(
    "/rds/rds-lxu/flamingo/L2p8_m9/lightcone0/healpix_map/rotation_groups"
)
META_PATH = MAP_DIR / "rotation_groups_L2p8_m9_lc0.json"
FIG_DIR = REPO / "figures/nb39_l2p8_rotation_group_tsz_ps"
FIG_DIR.mkdir(parents=True, exist_ok=True)

LMAX = 6000
N_Z_BINS = 60
A10_B1 = dict(P0=8.403, c500=1.177, gamma=0.3081, alpha=1.0510, beta=5.4905)

In [ ]:
def measure_dl_map(y, lmax=LMAX):
    nside = hp.npix2nside(y.size)
    fsky = float(np.mean(y != 0.0))
    cl = hp.anafast(y - y.mean(), lmax=lmax, iter=0)
    ell = np.arange(cl.size)
    cl = cl / (hp.pixwin(nside, lmax=lmax) ** 2 * fsky)
    return ell, ell * (ell + 1) / (2 * np.pi) * cl


def log_bin(ell, dl, lmin=10, lmax=LMAX, nbin=30):
    edges = np.logspace(np.log10(lmin), np.log10(lmax), nbin + 1)
    idx = np.digitize(ell, edges) - 1
    lb, db = [], []
    for b in range(nbin):
        sel = (idx == b) & (ell >= lmin)
        if sel.any():
            lb.append(ell[sel].mean())
            db.append(dl[sel].mean())
    return np.array(lb), np.array(db)


hm = HaloModel(cosmology=D3A_COSMOLOGY)
ell_th = jnp.logspace(1.0, np.log10(LMAX), 40)
m_grid = jnp.logspace(11.0, 15.5, 60)
pref = np.asarray(ell_th) * (np.asarray(ell_th) + 1) / (2 * np.pi)


def theory_dl(zlo, zhi, n_z_bins=N_Z_BINS):
    """tSZ D_l from hmfast (nb05): A10 GNFW B=1, geomspace z over [zlo, zhi]."""
    z = jnp.geomspace(max(zlo, 0.001), zhi, n_z_bins)
    profile = GNFWPressureProfile(**A10_B1, B=1.0)
    tracer = tSZTracer(profile=profile)
    cl1 = np.asarray(hm.cl_1h(tracer, tracer, l=ell_th, m=m_grid, z=z))
    cl2 = np.asarray(hm.cl_2h(tracer, tracer, l=ell_th, m=m_grid, z=z))
    return pref * (cl1 + cl2)


def rot_label(theta, phi):
    if theta == 0.0 and phi == 0.0:
        return "identity"
    return rf"$\theta={theta:.3f}$, $\phi={phi:.3f}$"

In [ ]:
meta = json.loads(META_PATH.read_text())
results = []

for g in meta["groups"]:
    gi = g["group"]
    zlo, zhi = g["z_inner"], g["z_outer"]
    fpath = MAP_DIR / g["fits"]
    y = hp.read_map(str(fpath), dtype=np.float64)
    ell, dl = measure_dl_map(y)
    ellb, dlb = log_bin(ell, dl)
    dth = theory_dl(zlo, zhi)
    results.append(dict(
        group=gi,
        shells=g["shells"],
        zlo=zlo,
        zhi=zhi,
        theta=g["rot_theta_rad"],
        phi=g["rot_phi_rad"],
        sum_y=float(y.sum()),
        y=y,
        ellb=ellb,
        dlb=dlb,
        dth=dth,
    ))
    d300_m = np.interp(3000, ell, dl)
    d300_t = np.interp(3000, np.asarray(ell_th), dth)
    print(
        f"group {gi}  shells {g['shell_first']}-{g['shell_last']}  "
        f"z=[{zlo:.3f},{zhi:.3f}]  sum(y)={y.sum():.2f}"
    )
    print(f"  D_3000 meas={d300_m:.2e}  theory={d300_t:.2e}")

In [ ]:
colors = ["#2166ac", "#1b7837", "#762a83", "#d6604d", "#878787"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharex=True, sharey=True)
axes = axes.ravel()

for ax, r, col in zip(axes, results, colors):
    sh = f"{r['shells'][0]}-{r['shells'][-1]}"
    ax.plot(r["ellb"], r["dlb"], "o", ms=3.5, color=col, label="measured")
    ax.plot(
        np.asarray(ell_th), r["dth"], "-", lw=1.8, color=col,
        label="hmfast A10 B=1 (nb05)",
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(20, LMAX)
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$D_\ell$")
    ax.set_title(
        rf"group {r['group']}: shells {sh}"
        "\n"
        rf"$z \in [{r['zlo']:.2f}, {r['zhi']:.2f}]$, "
        + rot_label(r["theta"], r["phi"])
    )
    ax.legend(fontsize=7)

axes[-1].axis("off")
fig.suptitle(
    r"L2p8_m9 lc0: tSZ $D_\ell$ per rotation-group shell sum vs hmfast (A10 GNFW, $B=1.0$)",
    y=1.01,
)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(FIG_DIR / f"rotation_group_tsz_ps_5panel.{ext}", bbox_inches="tight")
plt.show()
print("Saved to", FIG_DIR)

In [ ]:
# Overlay all groups on one axis (optional summary)
fig, ax = plt.subplots(figsize=(8, 5.5))
for r, col in zip(results, colors):
    sh = f"{r['shells'][0]}-{r['shells'][-1]}"
    ax.plot(
        r["ellb"], r["dlb"], "o", ms=3, color=col,
        label=rf"group {r['group']} (shells {sh})",
    )
    ax.plot(np.asarray(ell_th), r["dth"], "-", lw=1.5, color=col, alpha=0.85)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(20, LMAX)
ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$D_\ell$")
ax.set_title(r"L2p8_m9 lc0: rotation-group tSZ $D_\ell$ (points + theory lines)")
ax.legend(fontsize=8, ncol=1)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(FIG_DIR / f"rotation_group_tsz_ps_overlay.{ext}", bbox_inches="tight")
plt.show()

## Combined group sums

Pixel-wise sum of rotation-group maps (still **unrotated**):

1. **All groups 0–4** ($z \in [0.001, 3.0]$) vs hmfast over the full lightcone depth.
2. **Groups 1–4 only** (exclude identity group 0; shells 7–59, $z \in [0.35, 3.0]$) vs hmfast over the matching $z$ interval.

In [ ]:
def plot_combined(y, zlo, zhi, title, fname_stem):
    ell, dl = measure_dl_map(y)
    ellb, dlb = log_bin(ell, dl)
    dth = theory_dl(zlo, zhi)
    d300_m = np.interp(3000, ell, dl)
    d300_t = np.interp(3000, np.asarray(ell_th), dth)
    print(f"{title}: sum(y)={y.sum():.2f}  z=[{zlo:.3f},{zhi:.3f}]")
    print(f"  D_3000 meas={d300_m:.2e}  theory={d300_t:.2e}")

    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.plot(ellb, dlb, "o", ms=4, color="#2166ac", label="measured")
    ax.plot(
        np.asarray(ell_th), dth, "-", lw=1.8, color="#b2182b",
        label=rf"hmfast A10 B=1 (nb05, $z \in [{zlo:.2f},{zhi:.2f}]$)",
    )
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(20, LMAX)
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$D_\ell$")
    ax.set_title(title)
    ax.legend(fontsize=9)
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{fname_stem}.{ext}", bbox_inches="tight")
    plt.show()
    return dict(ellb=ellb, dlb=dlb, dth=dth, d300_m=d300_m, d300_t=d300_t)


y_all = sum(r["y"] for r in results)
z_all = (results[0]["zlo"], results[-1]["zhi"])
combo_all = plot_combined(
    y_all,
    *z_all,
    r"Sum of all rotation groups (0–4)",
    "rotation_group_sum_all_tsz_ps",
)

y_g14 = sum(r["y"] for r in results if r["group"] >= 1)
z_g14 = (results[1]["zlo"], results[-1]["zhi"])
combo_g14 = plot_combined(
    y_g14,
    *z_g14,
    r"Sum of groups 1–4 (shells 7–59, exclude group 0)",
    "rotation_group_sum_g1to4_tsz_ps",
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, combo, title in zip(
    axes,
    [combo_all, combo_g14],
    ["all groups 0–4", "groups 1–4 only"],
):
    ax.plot(combo["ellb"], combo["dlb"], "o", ms=3.5, color="#2166ac", label="measured")
    ax.plot(np.asarray(ell_th), combo["dth"], "-", lw=1.8, color="#b2182b", label="hmfast A10 B=1")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(20, LMAX)
    ax.set_xlabel(r"$\ell$")
    ax.set_ylabel(r"$D_\ell$")
    ax.set_title(title)
    ax.legend(fontsize=8)
fig.suptitle(r"L2p8_m9 lc0: combined rotation-group tSZ $D_\ell$", y=1.02)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(FIG_DIR / f"rotation_group_sum_combined_2panel.{ext}", bbox_inches="tight")
plt.show()
print("Saved combined figures to", FIG_DIR)